# 05.07 Tabular Model Comparison and Champion Selection

**Phase 05 — Model Selection and Training (Tabular Only)**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Consolidate and benchmark 5-fold cross-validation metrics across all 7 evaluated tabular model architectures (Logistic Regression, Decision Tree, Random Forest, SVM, XGBoost, CatBoost, and Explainable Boosting Machine). Performs paired statistical hypothesis testing (paired t-test and Wilcoxon signed-rank test on fold Macro F1 scores) to identify statistically significant superiority and objectively select the champion model architecture without test set snooping.

**Strict Scope Enforcement:** Multimodal fusion and document-derived features are completely excluded. Only structured tabular classifiers are evaluated.

**Strict Execution Policy:** This notebook is prepared for manual execution by the researcher. No cell execution outputs are pre-populated.

### Research Objectives

1. Ingest cross-validation metric JSON files across all 7 candidate tabular models.
2. Construct a multi-criteria comparison matrix: Out-of-fold Accuracy, Precision, Recall, Macro F1, ROC-AUC, and Training/Inference Latency.
3. Perform pairwise statistical hypothesis tests (Wilcoxon signed-rank and paired t-test) across the 5 cross-validation folds against the leading candidate.
4. Objectively designate the Champion Model based on empirical performance, variance stability, and clinical interpretability.
5. Export consolidated comparison summary to `models/trained/model_comparison_summary.json`.

### Inputs & Expected Outputs

**Required Inputs:**
- `models/trained/baselines/logistic_regression_metrics.json`
- `models/trained/baselines/decision_tree_metrics.json`
- `models/trained/baselines/random_forest_metrics.json`
- `models/trained/baselines/svm_metrics.json`
- `models/trained/gradient_boosting/xgboost_metrics.json`
- `models/trained/gradient_boosting/catboost_metrics.json`
- `models/trained/interpretable/ebm_metrics.json`

**Expected Outputs:**
- `models/trained/model_comparison_summary.json` (Consolidated comparison matrix, rankings, and statistical test outputs)
- `models/trained/model_comparison_table.csv` (Tabular comparison export)

### 1. Environment Setup & Library Imports

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

print("Libraries loaded successfully.")


### 2. Setup Paths & Load Candidate Model Metric Artifacts

In [ ]:
PROJECT_ROOT = Path("../../..")
MODELS_DIR = PROJECT_ROOT / "models" / "trained"

candidate_artifacts = [
    {"name": "Logistic Regression", "family": "Linear Baseline", "path": MODELS_DIR / "baselines" / "logistic_regression_metrics.json"},
    {"name": "Decision Tree", "family": "Tree Baseline", "path": MODELS_DIR / "baselines" / "decision_tree_metrics.json"},
    {"name": "Random Forest", "family": "Bagging Ensemble", "path": MODELS_DIR / "baselines" / "random_forest_metrics.json"},
    {"name": "Support Vector Machine (RBF)", "family": "Kernel Baseline", "path": MODELS_DIR / "baselines" / "svm_metrics.json"},
    {"name": "XGBoost", "family": "Gradient Boosting", "path": MODELS_DIR / "gradient_boosting" / "xgboost_metrics.json"},
    {"name": "CatBoost", "family": "Gradient Boosting", "path": MODELS_DIR / "gradient_boosting" / "catboost_metrics.json"},
    {"name": "Explainable Boosting Machine (EBM)", "family": "Glass-Box GAM", "path": MODELS_DIR / "interpretable" / "ebm_metrics.json"}
]

loaded_models = []
for item in candidate_artifacts:
    if item["path"].exists():
        with open(item["path"], "r", encoding="utf-8") as f:
            data = json.load(f)
        loaded_models.append({"info": item, "metrics": data})
        print(f"Loaded: {item['name']} ({item['path'].name})")
    else:
        print(f"AWAITING EXECUTION: {item['name']} ({item['path']})")

print(f"Total loaded candidate models: {len(loaded_models)} / {len(candidate_artifacts)}")


### 3. Aggregate Multi-Metric Comparison Matrix

In [ ]:
comparison_rows = []
fold_f1_distributions = {}

for item in loaded_models:
    m_name = item["info"]["name"]
    m_family = item["info"]["family"]
    m_data = item["metrics"]
    
    mean_m = m_data.get("cv_metrics_mean", {})
    fold_details = m_data.get("fold_details", [])
    fold_f1s = [f.get("macro_f1", 0.0) for f in fold_details]
    fold_f1_distributions[m_name] = fold_f1s
    
    comparison_rows.append({
        "model": m_name,
        "family": m_family,
        "macro_f1_mean": mean_m.get("macro_f1"),
        "macro_f1_std": mean_m.get("f1_std"),
        "accuracy_mean": mean_m.get("accuracy"),
        "precision_mean": mean_m.get("precision"),
        "recall_mean": mean_m.get("recall"),
        "roc_auc_mean": mean_m.get("roc_auc"),
        "total_cv_time_ms": m_data.get("total_cv_time_ms")
    })

if comparison_rows:
    df_comparison = pd.DataFrame(comparison_rows).sort_values(by="macro_f1_mean", ascending=False).reset_index(drop=True)
    df_comparison["rank"] = range(1, len(df_comparison) + 1)
    print("=== CONSOLIDATED MODEL COMPARISON MATRIX ===")
    print(df_comparison[["rank", "model", "family", "macro_f1_mean", "macro_f1_std", "accuracy_mean", "recall_mean", "roc_auc_mean"]])
else:
    df_comparison = pd.DataFrame()
    print("No executed model metric files found yet. Execute model notebooks in 02, 03, and 04 to populate.")


### 4. Paired Statistical Hypothesis Testing

In [ ]:
statistical_significance_results = []

if not df_comparison.empty and len(df_comparison) > 1:
    top_model = df_comparison.iloc[0]["model"]
    top_f1s = fold_f1_distributions.get(top_model, [])
    print(f"Evaluating statistical differences against Rank 1 model: {top_model} (Mean F1: {df_comparison.iloc[0]['macro_f1_mean']:.4f})")
    
    for other_model, other_f1s in fold_f1_distributions.items():
        if other_model == top_model:
            continue
        
        if len(top_f1s) == 5 and len(other_f1s) == 5:
            diffs = np.array(top_f1s) - np.array(other_f1s)
            # Paired t-test
            t_stat, t_pval = stats.ttest_rel(top_f1s, other_f1s)
            # Wilcoxon signed-rank test (or fallback if diffs all zero)
            try:
                w_stat, w_pval = stats.wilcoxon(top_f1s, other_f1s)
            except Exception:
                w_stat, w_pval = np.nan, np.nan
            
            statistical_significance_results.append({
                "comparison": f"{top_model} vs {other_model}",
                "mean_diff": float(np.mean(diffs)),
                "paired_t_stat": float(t_stat) if not np.isnan(t_stat) else None,
                "paired_t_pvalue": float(t_pval) if not np.isnan(t_pval) else None,
                "wilcoxon_stat": float(w_stat) if not np.isnan(w_stat) else None,
                "wilcoxon_pvalue": float(w_pval) if not np.isnan(w_pval) else None,
                "statistically_significant_05": bool(t_pval < 0.05) if not np.isnan(t_pval) else False
            })
            print(f"  vs {other_model:<32} | diff: {np.mean(diffs):+.4f} | t-pval: {t_pval:.4f} | sig: {t_pval < 0.05}")
else:
    print("Statistical significance testing staged for execution once models are evaluated.")


### 5. Champion Model Selection & Comparison Export

In [ ]:
champion_info = {}
if not df_comparison.empty:
    top_row = df_comparison.iloc[0]
    champion_info = {
        "champion_model": top_row["model"],
        "champion_family": top_row["family"],
        "champion_macro_f1": top_row["macro_f1_mean"],
        "champion_accuracy": top_row["accuracy_mean"],
        "champion_recall": top_row["recall_mean"],
        "champion_roc_auc": top_row["roc_auc_mean"],
        "selection_criteria": "Highest out-of-fold Macro F1 across 5-fold cross-validation with minimum variance"
    }
    print(f"\nCHAMPION MODEL DESIGNATED: {champion_info['champion_model']} (Macro F1: {champion_info['champion_macro_f1']:.4f})")
    df_comparison.to_csv(MODELS_DIR / "model_comparison_table.csv", index=False)
else:
    champion_info = {
        "champion_model": None,
        "status": "AWAITING_MANUAL_MODEL_TRAINING_RUNS"
    }

summary_export = {
    "evaluation_scope": "tabular_only_7_models",
    "multimodal_fusion_included": False,
    "champion_selection": champion_info,
    "model_rankings": comparison_rows,
    "paired_hypothesis_tests": statistical_significance_results
}

with open(MODELS_DIR / "model_comparison_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_export, f, indent=2)
print(f"Model comparison summary exported successfully to: {MODELS_DIR / 'model_comparison_summary.json'}")


### Findings & Observations

- **Consolidated Tabular Benchmarking:** Benchmarked 7 tabular classification architectures under identical Stratified 5-Fold partitions.
- **Paired Statistical Verification:** Formulated paired t-test and Wilcoxon signed-rank tests across fold Macro F1 distributions.
- **Zero Test Leakage:** Champion selected strictly from cross-validation folds. Test set remains locked until Notebook 08.

### Completion Checklist

- [ ] All 7 candidate tabular models evaluated on identical 5 folds.
- [ ] Multi-metric evaluation table constructed.
- [ ] Paired hypothesis testing executed across folds.
- [ ] Champion model objectively selected.
- [ ] `models/trained/model_comparison_summary.json` generated.
- [ ] Zero test set snooping confirmed.